# Support Ticket Classification

# Import Required Libraries

In [1]:
import re

import pandas as pd

from IPython.display import display
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

seed = 42

# Create Ticket Dataset

In [2]:
rows = [
    (1, "I CAN'T login to my account!!!", "login_issue"),
    (2, "My password reset link is not working", "login_issue"),
    (3, "The account is locked after several attempts", "login_issue"),
    (4, "I forgot my password and cannot sign in", "login_issue"),
    (5, "Login keeps rejecting my correct password", "login_issue"),
    (6, "I cannot access my profile after signing in", "login_issue"),
    (7, "The verification code never arrives", "login_issue"),
    (8, "Sign in shows an unknown account error", "login_issue"),

    (9, "Payment failed but money was deducted", "payment_issue"),
    (10, "My card was charged twice for one order", "payment_issue"),
    (11, "The transaction is still pending", "payment_issue"),
    (12, "Payment was rejected during checkout", "payment_issue"),
    (13, "The amount was debited but the order failed", "payment_issue"),
    (14, "Billing did not complete successfully", "payment_issue"),
    (15, "The checkout page keeps declining my card", "payment_issue"),
    (16, "I was charged even though payment failed", "payment_issue"),

    (17, "I need a refund for a broken product", "refund_request"),
    (18, "Please return my money for this damaged item", "refund_request"),
    (19, "How can I request a refund for my order?", "refund_request"),
    (20, "I cancelled the purchase and want my money back", "refund_request"),
    (21, "The product is defective and I need a refund", "refund_request"),
    (22, "Please tell me the status of my refund", "refund_request"),
    (23, "I want to return this order for a refund", "refund_request"),
    (24, "Can I get my payment refunded?", "refund_request"),

    (25, "<br> The app crashes after the update", "app_bug"),
    (26, "The application freezes on the home screen", "app_bug"),
    (27, "The upload button is not working", "app_bug"),
    (28, "The screen turns blank when I open settings", "app_bug"),
    (29, "The latest update broke the application", "app_bug"),
    (30, "The app closes whenever I upload a file", "app_bug"),
    (31, "Notifications stopped working after installation", "app_bug"),
    (32, "The mobile app becomes unresponsive", "app_bug"),
]

tickets = pd.DataFrame(
    rows,
    columns=["ticket_id", "raw_text", "category"],
)

display(tickets.head())
print("Total tickets:", len(tickets))

,ticket_id,raw_text,category
0,1,I CAN'T login to my account!!!,login_issue
1,2,My password reset link is not working,login_issue
2,3,The account is locked after several attempts,login_issue
3,4,I forgot my password and cannot sign in,login_issue
4,5,Login keeps rejecting my correct password,login_issue


Total tickets: 32


# Validate Dataset Quality

In [3]:
qualitySummary = pd.DataFrame({
    "missing_values": tickets.isna().sum(),
    "duplicate_values": [
        tickets["ticket_id"].duplicated().sum(),
        tickets["raw_text"].duplicated().sum(),
        tickets["category"].duplicated().sum(),
    ],
})

display(qualitySummary)
display(
    tickets["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="tickets")
)

,missing_values,duplicate_values
ticket_id,0,0
raw_text,0,0
category,0,28


,category,tickets
0,login_issue,8
1,payment_issue,8
2,refund_request,8
3,app_bug,8


# Clean Ticket Text

In [4]:
basicStopwords = {
    "the", "a", "an", "is", "are", "to", "for", "and", "or",
    "please", "hello", "hi", "thanks", "thank",
}

negationWords = {
    "no", "not", "never", "cannot", "cant", "dont",
}


def normalizeText(text):
    text = text.replace("can't", "cannot")
    text = text.replace("won't", "will not")
    text = text.replace("n't", " not")
    return text


def cleanText(text, removeStopwords=True):
    if not isinstance(text, str):
        return ""

    text = normalizeText(text.lower())
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    if removeStopwords:
        tokens = [
            token
            for token in tokens
            if token not in basicStopwords or token in negationWords
        ]

    return " ".join(tokens)


tickets["clean_text"] = tickets["raw_text"].map(cleanText)

display(
    tickets[
        ["raw_text", "clean_text"]
    ].head(8)
)

,raw_text,clean_text
0,I CAN'T login to my account!!!,i cannot login my account
1,My password reset link is not working,my password reset link not working
2,The account is locked after several attempts,account locked after several attempts
3,I forgot my password and cannot sign in,i forgot my password cannot sign in
4,Login keeps rejecting my correct password,login keeps rejecting my correct password
5,I cannot access my profile after signing in,i cannot access my profile after signing in
6,The verification code never arrives,verification code never arrives
7,Sign in shows an unknown account error,sign in shows unknown account error


# Demonstrate Word Stemming

In [5]:
stemmer = PorterStemmer()
sampleWords = [
    "running",
    "payments",
    "studies",
    "refunded",
    "crashes",
]

stemmingResults = pd.DataFrame({
    "word": sampleWords,
    "stem": [
        stemmer.stem(word)
        for word in sampleWords
    ],
})

display(stemmingResults)

,word,stem
0,running,run
1,payments,payment
2,studies,studi
3,refunded,refund
4,crashes,crash


# Split Training Data

In [6]:
XTrain, XTest, yTrain, yTest = train_test_split(
    tickets["raw_text"],
    tickets["category"],
    test_size=0.25,
    random_state=seed,
    stratify=tickets["category"],
)

print("Training tickets:", len(XTrain))
print("Testing tickets:", len(XTest))

Training tickets: 24
Testing tickets: 8


# Build Classification Pipeline

In [7]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            preprocessor=cleanText,
            ngram_range=(1, 2),
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=seed,
        ),
    ),
])

model.fit(XTrain, yTrain)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function cle...x7ea4b55eb590>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


# Evaluate Model Performance

In [8]:
predictions = model.predict(XTest)
accuracy = accuracy_score(yTest, predictions)

report = pd.DataFrame(
    classification_report(
        yTest,
        predictions,
        output_dict=True,
        zero_division=0,
    )
).transpose()

labels = sorted(tickets["category"].unique())
confusion = pd.DataFrame(
    confusion_matrix(
        yTest,
        predictions,
        labels=labels,
    ),
    index=labels,
    columns=labels,
)

print(f"Accuracy: {accuracy:.2%}")
display(report.round(3))
display(confusion)

Accuracy: 62.50%


,precision,recall,f1-score,support
app_bug,0.400,1.000,0.571,2.000
login_issue,0.000,0.000,0.000,2.000
payment_issue,1.000,0.500,0.667,2.000
refund_request,1.000,1.000,1.000,2.000
accuracy,0.625,0.625,0.625,0.625
macro avg,0.600,0.625,0.560,8.000
weighted avg,0.600,0.625,0.560,8.000


,app_bug,login_issue,payment_issue,refund_request
app_bug,2,0,0,0
login_issue,2,0,0,0
payment_issue,1,0,1,0
refund_request,0,0,0,2


# Predict New Tickets

In [9]:
newTickets = [
    "Money was deducted but checkout failed",
    "The app closes when I open my profile",
    "I cannot remember my password",
    "I want my money back for a damaged order",
]

predictionResults = pd.DataFrame({
    "ticket": newTickets,
    "predicted_category": model.predict(newTickets),
})

display(predictionResults)

,ticket,predicted_category
0,Money was deducted but checkout failed,payment_issue
1,The app closes when I open my profile,app_bug
2,I cannot remember my password,login_issue
3,I want my money back for a damaged order,refund_request
